In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
groq_api_key = os.getenv('GROQ_API_KEY')

In [6]:
import sys
!{sys.executable} -m pip install -U langchain-groq

  Attempting uninstall: langchain-groq
    Found existing installation: langchain-groq 0.1.5
    Uninstalling langchain-groq-0.1.5:
      Successfully uninstalled langchain-groq-0.1.5



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)

In [13]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016714553400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016714553850>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
from langchain_core.messages import HumanMessage, SystemMessage

message = [
    SystemMessage(content = "Translate the following from english to telugu and hindi"),
    HumanMessage(content = "Hello, How are you?")
]

In [20]:
response = llm.invoke(message)

In [21]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(response)

'**Telugu:**  \nహలో, మీరు ఎలా ఉన్నారు?  \n\n**Hindi:**  \nनमस्ते, आप कैसे हैं?'

In [23]:
## Using LCEL - to chain the components

chain = llm|parser
chain.invoke(message)

'**Telugu:**  \nహలో, మీరు ఎలా ఉన్నారు?  \n\n**Hindi:**  \nनमस्ते, आप कैसे हैं?'

In [30]:
## Prompt Templates

from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the text into {language}"

prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("user","{text}")
])

In [33]:
results = prompt.invoke({"language":"Telugu","text":"Is everyone okay?"})

In [36]:
results.to_messages()   # the same text we passed in earlier as human message and system message

[SystemMessage(content='Translate the text into Telugu', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Is everyone okay?', additional_kwargs={}, response_metadata={})]

In [37]:
chain = prompt|llm|parser

In [38]:
chain.invoke({"language":"Telugu","text":"Is everyone okay?"})

'అందరూ బాగున్నారా?'

In [40]:
from langchain_core.messages import HumanMessage

llm.invoke([HumanMessage(content="Hey I am Sumanth, I am AI/ML Engineer at Google")])

AIMessage(content='Nice to meet you, Sumanth! 👋 As an AI/ML Engineer at Google, you’re probably tackling some exciting projects. How can I help you today? Whether it’s brainstorming ideas, digging into ML concepts, or just chatting about the latest in AI, I’m here for you.', additional_kwargs={'reasoning_content': 'The user says "Hey I am Sumanth, I am AI/ML Engineer at Google". There\'s no question or request. The user is just introducing themselves. We should respond politely, maybe ask about how we can help or respond to the introduction. According to policy, we can respond. There\'s no disallowed content. So we can respond with friendly greeting.'}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 86, 'total_tokens': 230, 'completion_time': 0.151615584, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.034604165, 'prompt_tokens_details': None, 'queue_time': 0.149344573, 'total_time': 0.186219749}, 'model_name': 'openai/gpt-oss-20b'

In [42]:
from langchain_core.messages import AIMessage

llm.invoke([

    HumanMessage(content="I am Sumanth, an AI Engineer at Google from San Franscisco"),
    AIMessage(content="Hey Sumanth, Its really nice meeting you here, \n\nAs a AI Engineer what kind of work you do?"),
    HumanMessage(content="Who am I and what do I do?")
])

AIMessage(content='**Who you are**\n\n- **Name:** Sumanth  \n- **Profession:** AI Engineer  \n- **Employer:** Google  \n- **Location:** San\u202fFrancisco, California  \n\n**What you do**\n\nAs an AI Engineer at Google, you’re part of a team that designs, builds, and deploys machine‑learning solutions that power Google’s products and services. Your day‑to‑day responsibilities typically include:\n\n| Focus Area | Typical Tasks |\n|------------|---------------|\n| **Model Development** | • Build and train neural networks (e.g., transformers, CNNs, RNNs).<br>• Experiment with new architectures (e.g., Vision‑Transformer, BERT variants). |\n| **Data Engineering** | • Curate and preprocess large‑scale datasets (text, images, audio, video).<br>• Implement data pipelines with TensorFlow Data Services, BigQuery, or GCS. |\n| **Research & Innovation** | • Explore cutting‑edge research (e.g., self‑supervised learning, few‑shot learning).<br>• Publish findings in conferences (NeurIPS, ICML, CVPR).

## Message with Chat History

In [43]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(llm, get_session_history)

In [44]:
config = {"configurable" : {"session_id":"chat1"}}

In [48]:
response = with_message_history.invoke([HumanMessage(content="My name is Sumanth, AI Engineer at Google")],config=config)

In [49]:
response.content

'Nice to meet you, Sumanth! 👋 How can I assist you today? Whether it’s a technical question, a brainstorming session, or anything else, just let me know.'

In [52]:
config1 = {"configurable":{"session_id":"chat2"}}

In [55]:
with_message_history.invoke([HumanMessage(content="Who am I")],config=config)

AIMessage(content='You’re **Sumanth**, an AI Engineer at Google. If there’s anything specific you’d like to discuss—whether it’s AI projects, technical questions, or anything else—just let me know!', additional_kwargs={'reasoning_content': 'The user says "My name is Sumanth, AI Engineer at Google". They then ask "Who am I". The assistant should respond that the user is Sumanth, AI Engineer at Google. Probably also ask if they want more info.'}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 254, 'total_tokens': 356, 'completion_time': 0.110143181, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.012313661, 'prompt_tokens_details': None, 'queue_time': 0.003017261, 'total_time': 0.122456842}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_2b688e7cc3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13bf5-6af2-7fe0-9dca-7077253b32ab-0', tool_calls=[], inva

In [56]:
with_message_history.invoke([HumanMessage(content="Who am I")],config=config1)

AIMessage(content='I’m not sure what you’re hoping to uncover with the question “Who am I?”—are you looking for a quick, playful answer, a deeper philosophical reflection, or something that ties into a particular context (like a role you play, a project you’re working on, or a personal journey you’re on)? \n\nHere’s a short, reflective take that can apply to many situations:\n\n> **You are the sum of your experiences, your choices, and the stories you tell yourself and others.**  \n>  \n> In the moment, you’re a blend of memories, hopes, fears, and the small decisions that shape your day. Over time, those moments weave together into a narrative that’s uniquely yours—sometimes clear, sometimes messy, always evolving.\n\nIf you’d like to dive deeper—perhaps exploring your values, your passions, or the patterns in your life—I’d love to hear more about what you’re thinking. Feel free to share a bit of context, and we can explore it together!', additional_kwargs={'reasoning_content': 'The u

## Prompt Templates